# Lecture 03 — Image Processing / Обработка изображений

Практика RU/EN. Источник тем: исходный `ImageProcessing.ipynb`; адаптация 2026 — воспроизводимые данные, dtype/range, границы, исправленная FFT-маска.

Topics follow the original notebook. The 2026 adaptation adds reproducible data, explicit dtype/range and boundary contracts, and a corrected FFT mask. No downloads or credentials required.

Зависимости / Dependencies: `numpy`, `opencv-python-headless`, `matplotlib`. Restart & Run All.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
print("NumPy",np.__version__,"OpenCV",cv2.__version__)
rng=np.random.default_rng(20260927)
h,w=96,128
y,x=np.mgrid[:h,:w]
clean=np.clip(35+140*(x/w)+40*(y>h//2)+25*np.sin(2*np.pi*x/16),0,255).astype(np.uint8)
# A small bright structure / Маленькая яркая структура
clean[20:40,25:45]=220
assert clean.shape==(96,128) and clean.dtype==np.uint8


## 1. Контракт / Contract
float32 не означает диапазон [0,1]. / float32 does not imply a [0,1] range.
RGB/BGR относится к порядку каналов, а не типу данных. / RGB/BGR describes channel order, not dtype.


In [ ]:
a=np.array([250],np.uint8)
assert int((a+np.uint8(10))[0])==4
f=clean.astype(np.float32)/255
assert 0<=f.min()<=f.max()<=1
rgb=np.stack([clean,np.zeros_like(clean),255-clean],axis=-1)
bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
assert np.array_equal(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB),rgb)
fig,ax=plt.subplots(1,2,figsize=(9,3))
ax[0].imshow(rgb);ax[0].set_title('Correct RGB')
ax[1].imshow(bgr);ax[1].set_title('BGR displayed as RGB: wrong')
plt.tight_layout();plt.show()


## 2. Гистограмма / Histogram
Перемешивание сохраняет гистограмму и разрушает геометрию. / Shuffling preserves the histogram and destroys geometry.


In [ ]:
shuffled=rng.permutation(clean.ravel()).reshape(clean.shape)
hist=np.bincount(clean.ravel(),minlength=256)
assert hist.sum()==clean.size
assert np.array_equal(hist,np.bincount(shuffled.ravel(),minlength=256))
fig,ax=plt.subplots(1,3,figsize=(12,3))
ax[0].imshow(clean,cmap='gray',vmin=0,vmax=255);ax[0].set_title('Original')
ax[1].imshow(shuffled,cmap='gray',vmin=0,vmax=255);ax[1].set_title('Shuffled')
ax[2].plot(hist);ax[2].set_title('Same histogram');ax[2].set_xlabel('Intensity')
plt.tight_layout();plt.show()


## 3. Линейный фильтр и граница / Linear filter and boundary
`filter2D` вычисляет корреляцию; для симметричного ядра это также свёртка. / It computes correlation; a symmetric kernel also gives convolution.


In [ ]:
kernel=np.ones((3,3),np.float32)/9
constant=np.full((5,7),90,np.float32)
reflected=cv2.filter2D(constant,-1,kernel,borderType=cv2.BORDER_REFLECT_101)
zero=cv2.filter2D(constant,-1,kernel,borderType=cv2.BORDER_CONSTANT)
assert np.allclose(reflected,90)
assert np.isclose(zero[0,0],40)
print('Corner: reflected',reflected[0,0],'zero padded',zero[0,0])


## 4. Два шума и три фильтра / Two noise models and three filters
Сравниваем с чистым эталоном в общей шкале. / Compare against the clean reference with shared display limits.


In [ ]:
gaussian=np.clip(clean.astype(float)+rng.normal(0,18,clean.shape),0,255).astype(np.uint8)
impulse=clean.copy();u=rng.random(clean.shape);impulse[u<.04]=0;impulse[u>.96]=255
filters={'Input':lambda a:a,'Gaussian':lambda a:cv2.GaussianBlur(a,(5,5),1.2,borderType=cv2.BORDER_REFLECT_101),
         'Median':lambda a:cv2.medianBlur(a,3),
         'Bilateral':lambda a:cv2.bilateralFilter(a,7,25,3,borderType=cv2.BORDER_REFLECT_101)}
metrics={}
fig,ax=plt.subplots(2,4,figsize=(14,6))
for row,(label,noisy) in enumerate([('Gaussian noise',gaussian),('Impulse noise',impulse)]):
    metrics[label]={}
    for col,(method,fn) in enumerate(filters.items()):
        out=fn(noisy);mse=float(np.mean((out.astype(float)-clean.astype(float))**2))
        metrics[label][method]=mse
        ax[row,col].imshow(out,cmap='gray',vmin=0,vmax=255)
        ax[row,col].set_title(f'{label} / {method}\nMSE={mse:.1f}');ax[row,col].axis('off')
plt.tight_layout();plt.show()
print(metrics)
assert metrics['Impulse noise']['Median'] < metrics['Impulse noise']['Input']


## 5. FFT: полный цикл / Round trip
Логарифм спектра нужен для изображения, комплексные коэффициенты — для обратного преобразования. / The logarithmic spectrum is a display; complex coefficients are needed for reconstruction.


In [ ]:
F=np.fft.fftshift(np.fft.fft2(clean))
back=np.fft.ifft2(np.fft.ifftshift(F))
assert np.allclose(back.real,clean)
assert np.max(np.abs(back.imag))<1e-8
odd=clean[:71,:93];Fo=np.fft.fftshift(np.fft.fft2(odd))
assert np.allclose(np.fft.ifft2(np.fft.ifftshift(Fo)).real,odd)
plt.figure(figsize=(6,4));plt.imshow(np.log1p(np.abs(F)),cmap='magma');plt.title('log(1 + |FFT|)');plt.colorbar();plt.show()


## 6. High-pass: круг в индексах спектра / A circle in spectrum indices
В ДЗ сохраняется модуль и нормировка; здесь также смотрим знаковую действительную часть. / Homework retains magnitude and normalization; here we also inspect the signed real response.


In [ ]:
yy,xx=np.ogrid[:h,:w]
radius=8
mask=(yy-h//2)**2+(xx-w//2)**2 > radius**2
filtered=np.fft.ifft2(np.fft.ifftshift(F*mask))
assert mask[h//2,w//2]==0 and mask[0,0]==1
const=np.full((h,w),127,np.uint8)
Fc=np.fft.fftshift(np.fft.fft2(const))
assert np.max(np.abs(np.fft.ifft2(np.fft.ifftshift(Fc*mask))))<1e-8
fig,ax=plt.subplots(1,3,figsize=(12,3))
ax[0].imshow(mask,cmap='gray');ax[0].set_title('High-pass mask')
v=np.max(np.abs(filtered.real))
ax[1].imshow(filtered.real,cmap='coolwarm',vmin=-v,vmax=v);ax[1].set_title('Signed response')
ax[2].imshow(np.abs(filtered),cmap='gray');ax[2].set_title('Magnitude: sign lost')
plt.tight_layout();plt.show()


## 7. Пирамида / Pyramid
Сглаживание перед уменьшением ограничивает aliasing; оно не восстанавливает утраченную информацию. / Smoothing before downsampling limits aliasing; it does not restore lost information.


In [ ]:
levels=[clean]
for _ in range(3):levels.append(cv2.pyrDown(levels[-1]))
assert [a.shape for a in levels]==[(96,128),(48,64),(24,32),(12,16)]
fig,ax=plt.subplots(1,4,figsize=(12,3))
for axis,a in zip(ax,levels):axis.imshow(a,cmap='gray',vmin=0,vmax=255);axis.set_title(str(a.shape))
plt.tight_layout();plt.show()


## Самостоятельно / Your experiments

1. Измените seed, затем размер окна медианы. Разделите влияние этих изменений. / Change the seed and then the median window; separate their effects.
2. Повторите bilateral для float32 [0,1] с соответствующим sigmaColor. / Repeat bilateral on normalized float32 with scaled sigmaColor.
3. Возьмите прямоугольный массив нечётного размера и проверьте маску. / Test the mask on an odd rectangular array.
4. Опишите звон после жёсткой маски и сравните с плавной. / Describe ringing and compare a smooth transition.

## Источники / Sources
- Original instructor material: `ImageProcessing.ipynb`, repository snapshot `5856aa5`.
- https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html
- https://numpy.org/doc/stable/reference/routines.fft.html

## ДЗ / Homework
`Homework_RU_EN.ipynb` — 15 points; due 11 Oct 2026 23:59 Moscow.
